# Hands-on Lab: SMS Spam Detection using Naïve Bayes

**Dataset:** SMS Spam Collection (UCI)  

---

### Overview

This lab builds a spam filter using the Naïve Bayes algorithm on the classic SMS Spam Collection dataset. The pipeline covers text preprocessing, feature extraction (Bag of Words and TF-IDF), model training, evaluation, and a live classification function.

### Learning Objectives
- Understand why Naïve Bayes is effective for text classification
- Preprocess raw text: tokenisation, stop-word removal, vectorisation
- Recognise the difference between BoW and TF-IDF representations
- Evaluate a model correctly under class imbalance
- Deploy a live `classify_message()` function

## Step 1: Import Required Packages

In [ ]:
import re
import nltk
import pandas as pd
from nltk.corpus import stopwords
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

nltk.download('stopwords')

## Step 2: Load and Preprocess the Dataset

In [ ]:
data = pd.read_csv('SMSSpamCollection.csv')
print(data.head())
print(f"\nDataset shape: {data.shape}")
print(f"\nLabel distribution:\n{data['Label'].value_counts()}")

In [ ]:
data['Label'] = data['Label'].map({'spam': 1, 'ham': 0})

# Define stop words
stop_words = set(stopwords.words('english'))

def preprocess_text(text):
    """Clean a single message: lowercase, remove non-alpha, strip stop words."""
    text = text.lower()
    text = re.sub(r'[^a-z\s]', '', text)
    tokens = text.split()
    tokens = [word for word in tokens if word not in stop_words]
    return ' '.join(tokens)

data['cleaned_message'] = data['Message'].apply(preprocess_text)

print(data[['Message', 'cleaned_message']].head())

## Step 3: Train/Test Split

**Important:** the split happens *before* vectorisation. Fitting the vectoriser on the full dataset would leak test vocabulary into the feature space, artificially inflating performance metrics.

In [ ]:
X = data['cleaned_message']
y = data['Label']

# Split first — vectorise after
X_train_raw, X_test_raw, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print(f"Training set size: {X_train_raw.shape[0]} samples")
print(f"Testing set size:  {X_test_raw.shape[0]} samples")

## Step 4: Feature Extraction

### Part 1 — Bag of Words (BoW)

BoW counts how many times each word appears in a message. Simple but effective for Naïve Bayes since word frequency is a strong spam signal.

In [ ]:
vectorizer = CountVectorizer()

X_train_bow = vectorizer.fit_transform(X_train_raw)
X_test_bow  = vectorizer.transform(X_test_raw)

# Preview BoW matrix (first 5 rows, first 10 features)
bow_df = pd.DataFrame(
    X_train_bow.toarray(),
    columns=vectorizer.get_feature_names_out()
)
print("Bag of Words (BoW) — training set preview:")
print(bow_df.iloc[:5, :10])

### Part 2 — TF-IDF

TF-IDF down-weights words that appear frequently across all messages (e.g. "free" appearing in every spam) and up-weights words that are distinctive to specific messages. Often outperforms BoW for spam detection.

In [ ]:
tfidf_vectorizer = TfidfVectorizer()

# Fit ONLY on training data
X_train_tfidf = tfidf_vectorizer.fit_transform(X_train_raw)
X_test_tfidf  = tfidf_vectorizer.transform(X_test_raw)

# Preview TF-IDF matrix
tfidf_df = pd.DataFrame(
    X_train_tfidf.toarray(),
    columns=tfidf_vectorizer.get_feature_names_out()
)
print("TF-IDF — training set preview:")
print(tfidf_df.iloc[:5, :10])

## Step 5: Train Naïve Bayes Model

Training on BoW features. MultinomialNB is the natural fit here — it is designed for discrete count data, which is exactly what BoW produces.

In [ ]:
model = MultinomialNB()
model.fit(X_train_bow, y_train)

## Step 6: Evaluate the Model

In [ ]:
y_pred = model.predict(X_test_bow)

accuracy = accuracy_score(y_test, y_pred)
print(f"Accuracy: {accuracy:.2f}")

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred))

print("\nClassification Report:")
print(classification_report(y_test, y_pred, target_names=['ham', 'spam']))

## Step 7: Live Classification Function

In [ ]:
def classify_message(message):
    """Classify a single raw SMS message as Spam or Ham."""
    cleaned  = preprocess_text(message)
    vec_msg  = vectorizer.transform([cleaned])
    # prediction is an array — use [0] to extract the scalar value
    prediction = model.predict(vec_msg)
    return "Spam" if prediction[0] == 1 else "Ham"

# Example — known spam message
test_message = (
    "Thanks for your subscription to Ringtone UK your mobile will be "
    "charged £5/month Please confirm by replying YES or NO. "
    "If you reply NO you will not be charged"
)
print(f"Message: {test_message}")
print(f"Classification: {classify_message(test_message)}")

# Test with a ham message
ham_message = "Hey, are we still meeting for lunch tomorrow?"
print(f"\nMessage: {ham_message}")
print(f"Classification: {classify_message(ham_message)}")

---

## Personal Analysis

### Class Imbalance and Why Accuracy Is Misleading

The dataset contains 4,827 ham messages and only 747 spam messages — an ~87/13 split. A naive classifier that labels everything as ham would achieve 87% accuracy without learning anything. This is why **precision, recall, and F1-score on the spam class** are the metrics that actually matter here:

- **Precision (spam):** of all messages flagged as spam, how many actually were? Low precision means legitimate messages get blocked — a bad user experience.
- **Recall (spam):** of all actual spam messages, how many were caught? Low recall means spam slips through — the filter fails its primary job.
- The F1-score balances both. For a spam filter, recall is typically prioritised over precision — missing spam is worse than the occasional false positive.

### Why Naïve Bayes Works Well for This Task

Despite its "naïve" independence assumption (that each word's presence is independent of every other word), MultinomialNB performs strongly on text classification because:
- Spam messages genuinely do contain distinctive vocabulary ("free", "win", "prize", "urgent") that appears at very different frequencies compared to ham
- The independence assumption, while technically wrong, rarely hurts in practice for bag-of-words features
- Training is extremely fast even on large vocabularies, making it practical for real-time filtering

### BoW vs TF-IDF

This lab trains on BoW features. TF-IDF would likely perform comparably or slightly better here because it down-weights words like "call" or "get" that appear frequently across both spam and ham, giving more weight to words that are truly distinctive. A natural extension would be to train two models side by side and compare their F1-scores on the spam class directly.

### The Data Leakage Issue

A common mistake in text classification pipelines is fitting the vectoriser on the full dataset before splitting. This causes the vectoriser's vocabulary to include words from test messages, which means the model has indirect knowledge of the test set during training — inflating reported performance. The correct pattern, used in this notebook, is: **split first, then fit the vectoriser on training data only**, and use `transform()` (not `fit_transform()`) on the test set.